# MIMIC Property Graph Pipeline: Pre-UMLS then UMLS

This notebook runs offline vLLM inference in-process and executes one configurable experiment in two stages:

1. `pre_umls`: fast graph construction without UMLS normalization.
2. `umls`: final graph construction with UMLS entity standardization and concept hints.

Set provider, model sweep, note limits, and UMLS controls through environment variables or the setup cell. Use `all`, `none`, or `unlimited` for no cap on note count or note length.

In [ ]:
from __future__ import annotations

import importlib.util
import sys

# Notebook dependency bootstrap. Install binary dependencies first because Viper's
# system compiler is too old to build current greenlet releases from source.
binary_packages = {
    "greenlet": "greenlet==3.1.1",
}

required_packages = {
    "pandas": "pandas==2.3.3",
    "requests": "requests==2.33.1",
    "dotenv": "python-dotenv==1.0.1",
    "llama_index.core": "llama-index-core",
    "vllm": "vllm",
    "pyvis": "pyvis==0.3.2",
    "matplotlib": "matplotlib",
    "networkx": "networkx",
    "yaml": "PyYAML==6.0.3",
    "bs4": "beautifulsoup4==4.14.3",
}

def is_installed(module_name: str) -> bool:
    try:
        return importlib.util.find_spec(module_name) is not None
    except ModuleNotFoundError:
        return False

missing_binary_packages = [package for module, package in binary_packages.items() if not is_installed(module)]
missing_packages = [package for module, package in required_packages.items() if not is_installed(module)]

ipython = get_ipython()
if (missing_binary_packages or missing_packages) and ipython is None:
    raise RuntimeError("Run this cell in Jupyter/IPython so %pip installs into the active kernel.")

if missing_binary_packages:
    print("Installing required binary wheels:", missing_binary_packages)
    try:
        ipython.run_line_magic(
            "pip",
            "install --only-binary=:all: " + " ".join(missing_binary_packages),
        )
    except Exception as exc:
        raise RuntimeError(
            "Viper could not obtain a prebuilt greenlet wheel. Do not compile it with the "
            "system GCC; ask the environment administrator to provide greenlet==3.1.1."
        ) from exc

if missing_packages:
    print("Installing missing notebook dependencies:", missing_packages)
    ipython.run_line_magic("pip", "install " + " ".join(missing_packages))
    print("Install complete. Restart the kernel if imports still fail.")
else:
    print("All notebook dependencies are already installed for", sys.executable)


## Notebook Settings

Edit this cell for Viper. The notebook defines its own variables here and does not use a `.env` file for configuration.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

# Core experiment settings. Edit these values in this cell; no .env file is used.
EXPERIMENT_NAME = "mimic_umls_pipeline"
MIMIC_DISCHARGE_CSV = "data/mimic_iv_note/discharge.csv"
EXPERIMENT_INPUT_DIR = f"data/evidence/{EXPERIMENT_NAME}"
EXPERIMENT_OUTPUT_DIR = f"output/{EXPERIMENT_NAME}"
MIMIC_DISCHARGE_NOTE_TYPE = "DS"
MIMIC_DISCHARGE_LIMIT = "all"
MIMIC_DISCHARGE_MAX_CHARS = "3000"
EVAL_SAMPLE_SIZE = 1
NOTEBOOK_GRAPH_PREVIEW_LIMIT = 6
NOTEBOOK_INTERACTIVE_GRAPH_INDEX = 0

# Offline vLLM settings for Viper. The notebook kernel must run on the allocated GPU node.
VLLM_MODEL = "Qwen/Qwen2.5-3B-Instruct"
VLLM_EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
VLLM_TENSOR_PARALLEL_SIZE = 1
VLLM_EMBEDDING_TENSOR_PARALLEL_SIZE = 1
VLLM_GPU_MEMORY_UTILIZATION = 0.60
VLLM_EMBEDDING_GPU_MEMORY_UTILIZATION = 0.20
VLLM_EMBEDDING_FALLBACK_DEVICE = "cuda"
VLLM_MAX_MODEL_LEN = "4096"
# Leave blank to use each embedding model's native context limit. BGE v1.5 uses 512.
VLLM_EMBEDDING_MAX_MODEL_LEN = ""
VLLM_MAX_TOKENS = 384
VLLM_TEMPERATURE = 0.1
VLLM_TOP_P = 0.95
VLLM_TRUST_REMOTE_CODE = False
INDEX_LLM_REQUEST_TIMEOUT = 240
MPLCONFIGDIR = str(Path("output") / ".matplotlib")

# Keep model downloads off the small HPC home quota. SLURM_TMPDIR is preferred when available.
MODEL_CACHE_ROOT = Path(
    os.environ.get("SLURM_TMPDIR")
    or os.environ.get("TMPDIR")
    or f"/tmp/{os.environ.get('USER', 'vllm')}"
) / "huggingface"
MODEL_CACHE_MIN_FREE_GB = 8

# Text-only generation models supported by vLLM. Keep only a small number selected:
# generation models x embedding models x graph stages determines the number of full builds.
TEXT_GENERATION_MODELS = [
    {"model": "Qwen/Qwen2.5-3B-Instruct", "family": "Qwen2.5", "size": "3B", "gpu_20gb": "comfortable", "access": "open", "selected": True, "notes": "Recommended default while an embedding engine shares the GPU"},
    {"model": "microsoft/Phi-3.5-mini-instruct", "family": "Phi 3.5", "size": "3.8B", "gpu_20gb": "comfortable", "access": "open", "selected": False, "notes": "Compact long-context baseline"},
    {"model": "HuggingFaceTB/SmolLM2-1.7B-Instruct", "family": "SmolLM2", "size": "1.7B", "gpu_20gb": "comfortable", "access": "open", "selected": False, "notes": "Very fast small-model baseline"},
    {"model": "Qwen/Qwen2.5-7B-Instruct", "family": "Qwen2.5", "size": "7B", "gpu_20gb": "tight", "access": "open", "selected": False, "notes": "May fit at 4K context, but leaves little room for the embedding engine"},
    {"model": "mistralai/Mistral-7B-Instruct-v0.3", "family": "Mistral", "size": "7B", "gpu_20gb": "tight", "access": "open", "selected": False, "notes": "Test alone before a full graph build"},
    {"model": "meta-llama/Llama-3.1-8B-Instruct", "family": "Llama 3.1", "size": "8B", "gpu_20gb": "quantized", "access": "gated", "selected": False, "notes": "Use a supported quantized checkpoint; Hugging Face access required"},
    {"model": "google/gemma-2-9b-it", "family": "Gemma 2", "size": "9B", "gpu_20gb": "quantized", "access": "gated", "selected": False, "notes": "Use a supported quantized checkpoint; Hugging Face access required"},
    {"model": "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B", "family": "DeepSeek R1 Distill", "size": "7B", "gpu_20gb": "tight", "access": "open", "selected": False, "notes": "Reasoning output increases runtime and token use"},
    {"model": "ibm-granite/granite-3.3-8b-instruct", "family": "Granite 3.3", "size": "8B", "gpu_20gb": "quantized", "access": "open", "selected": False, "notes": "Newer vLLM and quantization recommended"},
    {"model": "Qwen/Qwen3-8B", "family": "Qwen3", "size": "8B", "gpu_20gb": "quantized", "access": "open", "selected": False, "notes": "Newer vLLM and quantization recommended"},
    {"model": "Qwen/Qwen3-14B", "family": "Qwen3", "size": "14B", "gpu_20gb": "not recommended", "access": "open", "selected": False, "notes": "Use a larger or multi-GPU allocation"},
    {"model": "mistralai/Mistral-Nemo-Instruct-2407", "family": "Mistral NeMo", "size": "12B", "gpu_20gb": "not recommended", "access": "open", "selected": False, "notes": "Use a larger or multi-GPU allocation"},
    {"model": "Qwen/Qwen2.5-14B-Instruct", "family": "Qwen2.5", "size": "14B", "gpu_20gb": "not recommended", "access": "open", "selected": False, "notes": "Use a larger or multi-GPU allocation"},
    {"model": "Qwen/Qwen3-30B-A3B", "family": "Qwen3 MoE", "size": "30B / 3B active", "gpu_20gb": "not recommended", "access": "open", "selected": False, "notes": "Total weights, rather than active parameters, determine fit"},
    {"model": "meta-llama/Llama-3.1-70B-Instruct", "family": "Llama 3.1", "size": "70B", "gpu_20gb": "multi-GPU", "access": "gated", "selected": False, "notes": "Not suitable for one 20 GB GPU"},
]
GENERATION_MODEL_SWEEP = [item["model"] for item in TEXT_GENERATION_MODELS if item["selected"]]

# All entries below are text-only pooling models. Set selected=True to add a model.
# Large models are opt-in because
# every selected encoder rebuilds each graph stage and must coexist with the generation model.
TEXT_EMBEDDING_MODELS = [
    {"model": "Qwen/Qwen3-Embedding-0.6B", "size": "0.6B", "languages": "multilingual", "selected": False, "notes": "Strong compact Qwen baseline; newer vLLM required"},
    {"model": "google/embeddinggemma-300m", "size": "0.3B", "languages": "multilingual", "selected": False, "notes": "Compact modern baseline; newer vLLM required"},
    {"model": "BAAI/bge-small-en-v1.5", "size": "33M", "languages": "English", "selected": True, "notes": "Fast BGE baseline"},
    {"model": "BAAI/bge-base-en-v1.5", "size": "109M", "languages": "English", "selected": False, "notes": "Balanced BGE baseline"},
    {"model": "intfloat/e5-base-v2", "size": "109M", "languages": "English", "selected": False, "notes": "Popular retrieval baseline"},
    {"model": "Alibaba-NLP/gte-modernbert-base", "size": "149M", "languages": "English", "selected": False, "notes": "Long-context ModernBERT encoder"},
    {"model": "Snowflake/snowflake-arctic-embed-m-v1.5", "size": "109M", "languages": "English", "selected": False, "notes": "Retrieval-focused compact model"},
    {"model": "nomic-ai/nomic-embed-text-v1.5", "size": "137M", "languages": "English", "selected": False, "notes": "Long-context; may require trust_remote_code"},
    {"model": "BAAI/bge-large-en-v1.5", "size": "335M", "languages": "English", "selected": False, "notes": "Higher-quality BGE; slower"},
    {"model": "intfloat/e5-large-v2", "size": "335M", "languages": "English", "selected": False, "notes": "Higher-quality E5; slower"},
    {"model": "Qwen/Qwen3-Embedding-4B", "size": "4B", "languages": "multilingual", "selected": False, "notes": "Large Qwen; opt in on 40 GB GPU"},
    {"model": "Qwen/Qwen3-Embedding-8B", "size": "8B", "languages": "multilingual", "selected": False, "notes": "Largest Qwen; test separately"},
    {"model": "BAAI/bge-m3", "size": "568M", "languages": "multilingual", "selected": False, "notes": "Dense output used; sparse/ColBERT weights are extra"},
    {"model": "intfloat/multilingual-e5-large", "size": "560M", "languages": "multilingual", "selected": False, "notes": "Multilingual E5 baseline"},
    {"model": "jinaai/jina-embeddings-v3", "size": "570M", "languages": "multilingual", "selected": False, "notes": "vLLM currently uses its text-matching task"},
]
EMBEDDING_MODEL_SWEEP = [item["model"] for item in TEXT_EMBEDDING_MODELS if item["selected"]]

# Pipeline controls. The UMLS stage needs UMLS_API_KEY.
RUN_PRE_UMLS = True
RUN_UMLS = True
PRE_UMLS_SCHEMA_GUIDED = False
UMLS_SCHEMA_GUIDED = False
UMLS_API_KEY = ""
UMLS_VERSION = "current"
UMLS_BASE_URL = "https://uts-ws.nlm.nih.gov/rest"
UMLS_PAGE_SIZE = 10
UMLS_HINT_LIMIT = 120
UMLS_SOURCE_VOCABS = "ICD10CM,SNOMEDCT_US,RXNORM,ATC,CPT,HCPCS,LNC,MEDCIN,MSH"

# Mirror notebook variables into os.environ only because the repo's lower-level
# index helpers read their runtime settings from environment variables.
notebook_env = {
    "EXPERIMENT_NAME": EXPERIMENT_NAME,
    "MIMIC_DISCHARGE_CSV": MIMIC_DISCHARGE_CSV,
    "EXPERIMENT_INPUT_DIR": EXPERIMENT_INPUT_DIR,
    "EXPERIMENT_OUTPUT_DIR": EXPERIMENT_OUTPUT_DIR,
    "MIMIC_DISCHARGE_NOTE_TYPE": MIMIC_DISCHARGE_NOTE_TYPE,
    "MIMIC_DISCHARGE_LIMIT": MIMIC_DISCHARGE_LIMIT,
    "MIMIC_DISCHARGE_MAX_CHARS": MIMIC_DISCHARGE_MAX_CHARS,
    "EVAL_SAMPLE_SIZE": str(EVAL_SAMPLE_SIZE),
    "NOTEBOOK_GRAPH_PREVIEW_LIMIT": str(NOTEBOOK_GRAPH_PREVIEW_LIMIT),
    "NOTEBOOK_INTERACTIVE_GRAPH_INDEX": str(NOTEBOOK_INTERACTIVE_GRAPH_INDEX),
    "INDEX_LLM_PROVIDER": "vllm_offline",
    "INDEX_EMBEDDING_PROVIDER": "vllm_offline",
    "INDEX_LLM_REQUEST_TIMEOUT": str(INDEX_LLM_REQUEST_TIMEOUT),
    "VLLM_MODEL_SWEEP": ",".join(GENERATION_MODEL_SWEEP),
    "VLLM_MODEL": VLLM_MODEL,
    "VLLM_EMBEDDING_MODEL": VLLM_EMBEDDING_MODEL,
    "VLLM_TENSOR_PARALLEL_SIZE": str(VLLM_TENSOR_PARALLEL_SIZE),
    "VLLM_EMBEDDING_TENSOR_PARALLEL_SIZE": str(VLLM_EMBEDDING_TENSOR_PARALLEL_SIZE),
    "VLLM_GPU_MEMORY_UTILIZATION": str(VLLM_GPU_MEMORY_UTILIZATION),
    "VLLM_EMBEDDING_GPU_MEMORY_UTILIZATION": str(VLLM_EMBEDDING_GPU_MEMORY_UTILIZATION),
    "VLLM_EMBEDDING_FALLBACK_DEVICE": VLLM_EMBEDDING_FALLBACK_DEVICE,
    "VLLM_MAX_MODEL_LEN": str(VLLM_MAX_MODEL_LEN),
    "VLLM_EMBEDDING_MAX_MODEL_LEN": str(VLLM_EMBEDDING_MAX_MODEL_LEN),
    "VLLM_MAX_TOKENS": str(VLLM_MAX_TOKENS),
    "VLLM_TEMPERATURE": str(VLLM_TEMPERATURE),
    "VLLM_TOP_P": str(VLLM_TOP_P),
    "VLLM_TRUST_REMOTE_CODE": str(VLLM_TRUST_REMOTE_CODE).lower(),
    "VLLM_EMBEDDING_MODEL_SWEEP": ",".join(EMBEDDING_MODEL_SWEEP),
    "RUN_PRE_UMLS": str(RUN_PRE_UMLS).lower(),
    "RUN_UMLS": str(RUN_UMLS).lower(),
    "PRE_UMLS_SCHEMA_GUIDED": str(PRE_UMLS_SCHEMA_GUIDED).lower(),
    "UMLS_SCHEMA_GUIDED": str(UMLS_SCHEMA_GUIDED).lower(),
    "UMLS_VERSION": UMLS_VERSION,
    "UMLS_BASE_URL": UMLS_BASE_URL,
    "UMLS_PAGE_SIZE": str(UMLS_PAGE_SIZE),
    "UMLS_HINT_LIMIT": str(UMLS_HINT_LIMIT),
    "UMLS_SOURCE_VOCABS": UMLS_SOURCE_VOCABS,
    "MPLCONFIGDIR": MPLCONFIGDIR,
    "HF_HOME": str(MODEL_CACHE_ROOT),
    "HUGGINGFACE_HUB_CACHE": str(MODEL_CACHE_ROOT / "hub"),
    "TRANSFORMERS_CACHE": str(MODEL_CACHE_ROOT / "transformers"),
}
if UMLS_API_KEY:
    notebook_env["UMLS_API_KEY"] = UMLS_API_KEY
else:
    os.environ.pop("UMLS_API_KEY", None)

os.environ.update(notebook_env)

print("Notebook variables configured; no .env file is used.")
print("VLLM_MODEL:", VLLM_MODEL)
print("VLLM_EMBEDDING_MODEL:", VLLM_EMBEDDING_MODEL)
print("MODEL_CACHE_ROOT:", MODEL_CACHE_ROOT)
print("MIMIC_DISCHARGE_LIMIT:", MIMIC_DISCHARGE_LIMIT)


In [ ]:
from __future__ import annotations

import asyncio
import gc
import inspect
import os
import shutil
import sys
import time
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import pandas as pd
from IPython.display import Image, IFrame, display

repo_root = Path.cwd()
if not (repo_root / "main.py").exists():
    for parent in Path.cwd().resolve().parents:
        if (parent / "main.py").exists():
            repo_root = parent
            break

sys.path.insert(0, str(repo_root))

from eval.medqa_smoke import load_questions, format_options, extract_answer
from helpers.config import parse_optional_int
from ingest.mimic import MimicDischargeSubsetConfig, extract_mimic_discharge_subset
from rag.index import build_embed_model, build_llm, ensure_index
from rag.retrieve import query_index_context
from rag.visualize import save_clinical_entity_graph, save_clinical_entity_graph_jpeg

# The notebook and offline backend must be deployed together. Catch stale HPC copies
# before preparing all MIMIC notes or starting an expensive graph build.
offline_backend_path = repo_root / "rag" / "vllm_offline.py"
offline_backend_missing = not offline_backend_path.exists()
offline_llm_missing = 'provider == "vllm_offline"' not in inspect.getsource(build_llm)
offline_embedding_missing = 'provider == "vllm_offline"' not in inspect.getsource(build_embed_model)
if offline_backend_missing or offline_llm_missing or offline_embedding_missing:
    raise RuntimeError(
        "This HPC checkout has a stale offline-vLLM backend. Synchronize rag/index.py, "
        "rag/vllm_offline.py, helpers/config.py, and this notebook from the same repo revision, "
        "then restart the kernel. Do not change INDEX_LLM_PROVIDER to 'vllm'; that mode expects "
        "an HTTP server."
    )

MODEL_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
cache_free_gb = shutil.disk_usage(MODEL_CACHE_ROOT).free / (1024 ** 3)
print(f"model_cache_free_gb: {cache_free_gb:.1f} at {MODEL_CACHE_ROOT}")
if cache_free_gb < MODEL_CACHE_MIN_FREE_GB:
    raise RuntimeError(
        f"Only {cache_free_gb:.1f} GB is free in MODEL_CACHE_ROOT={MODEL_CACHE_ROOT}. "
        f"The selected models need at least {MODEL_CACHE_MIN_FREE_GB} GB free. Set "
        "MODEL_CACHE_ROOT in the settings cell to a larger Viper scratch or BeegFS path, "
        "restart the kernel, and rerun from the first cell."
    )

experiment_name = EXPERIMENT_NAME
provider = "vllm_offline"
embedding_provider = "vllm_offline"

os.environ["INDEX_LLM_PROVIDER"] = "vllm_offline"
os.environ["INDEX_EMBEDDING_PROVIDER"] = "vllm_offline"

def model_slug(value: object | None) -> str:
    text = "unknown" if value is None else str(value)
    return "".join(ch if ch.isalnum() or ch in ("-", "_") else "_" for ch in text).strip("_") or "unknown"

def combo_slug(generation_model: str, embedding_model: str) -> str:
    return f"gen-{model_slug(generation_model)}__embed-{model_slug(embedding_model)}"

default_generation_models = [item["model"] for item in TEXT_GENERATION_MODELS if item["selected"]]
default_embedding_models = [item["model"] for item in TEXT_EMBEDDING_MODELS if item["selected"]]

generation_model_sweep = list(GENERATION_MODEL_SWEEP or default_generation_models)
embedding_model_sweep = list(EMBEDDING_MODEL_SWEEP or default_embedding_models)

if not generation_model_sweep:
    raise ValueError("No vLLM generation models configured.")
if not embedding_model_sweep:
    raise ValueError("No vLLM embedding models configured.")

try:
    installed_vllm_version = version("vllm")
except PackageNotFoundError:
    installed_vllm_version = "not installed"

if installed_vllm_version.startswith("0.6."):
    print("WARNING: vLLM", installed_vllm_version, "is older than the current model catalogs.")
    print("Compact BERT-based embeddings will use Transformers locally because vLLM 0.6 cannot load BertModel.")
    print("Qwen2.5, Mistral 7B, Llama 3.1 and Gemma 2 are the safer generation choices in that environment.")
    print("Qwen3, newer Granite models, Qwen3-Embedding and EmbeddingGemma may require a newer Viper environment.")

print("repo_root:", repo_root)
print("experiment_name:", experiment_name)
print("provider:", provider)
print("embedding_provider:", embedding_provider)
print("vllm_version:", installed_vllm_version)
print("generation_model_sweep:", generation_model_sweep)
print("embedding_model_sweep:", embedding_model_sweep)
model_combinations = len(generation_model_sweep) * len(embedding_model_sweep)
configured_stage_count = int(RUN_PRE_UMLS) + int(RUN_UMLS and bool(UMLS_API_KEY))
print("total_model_combinations:", model_combinations)
print("configured_graph_builds:", model_combinations * configured_stage_count)

generation_catalog = pd.DataFrame(TEXT_GENERATION_MODELS)
display(generation_catalog.style.apply(
    lambda row: ["font-weight: bold" if row["selected"] else "" for _ in row],
    axis=1,
))

embedding_catalog = pd.DataFrame(TEXT_EMBEDDING_MODELS)
display(embedding_catalog.style.apply(
    lambda row: ["font-weight: bold" if row["selected"] else "" for _ in row],
    axis=1,
))


In [ ]:
mimic_csv = Path(MIMIC_DISCHARGE_CSV)
mimic_notes_dir = Path(EXPERIMENT_INPUT_DIR)
output_root = Path(EXPERIMENT_OUTPUT_DIR)
output_root.mkdir(parents=True, exist_ok=True)

note_limit = parse_optional_int(MIMIC_DISCHARGE_LIMIT, None)
note_max_chars = parse_optional_int(MIMIC_DISCHARGE_MAX_CHARS, 3000)
note_type = MIMIC_DISCHARGE_NOTE_TYPE

test_jsonl_candidates = [
    repo_root / "test.jsonl",
    repo_root / "data" / "medqa" / "data_clean" / "questions" / "US" / "test.jsonl",
    repo_root / "data" / "eval" / "test.jsonl",
]
test_jsonl = next((path for path in test_jsonl_candidates if path.exists()), None)

print("mimic_csv:", mimic_csv)
print("mimic_notes_dir:", mimic_notes_dir)
print("output_root:", output_root)
print("note_limit:", note_limit if note_limit is not None else "all")
print("note_max_chars:", note_max_chars if note_max_chars is not None else "all")
print("note_type:", note_type)
print("test_jsonl:", test_jsonl)


In [ ]:
if not mimic_csv.exists():
    raise FileNotFoundError(f"Missing MIMIC discharge CSV: {mimic_csv}")

written_notes = extract_mimic_discharge_subset(
    MimicDischargeSubsetConfig(
        csv_path=mimic_csv,
        output_dir=mimic_notes_dir,
        limit=note_limit,
        note_type=note_type,
        max_chars=note_max_chars,
        overwrite=env_bool("EXPERIMENT_OVERWRITE_NOTES", True),
    )
)
print(f"Prepared {len(written_notes)} notes in: {mimic_notes_dir}")


In [ ]:
questions = []
if test_jsonl:
    questions = load_questions(test_jsonl, sample_size=EVAL_SAMPLE_SIZE)
print("question_count:", len(questions))
pd.DataFrame([{"id": q.get("id"), "question": q.get("question"), "answer": q.get("answer")} for q in questions])


In [ ]:
run_umls_stage = RUN_UMLS
if run_umls_stage and not UMLS_API_KEY:
    print("RUN_UMLS=true but UMLS_API_KEY is blank. The final UMLS-standardized stage will be skipped.")
    print("Set UMLS_API_KEY in the Notebook Settings cell to run the final standardized graph.")
    run_umls_stage = False

stages = [
    {"name": "pre_umls", "use_umls": False, "schema_guided": PRE_UMLS_SCHEMA_GUIDED, "enabled": RUN_PRE_UMLS},
    {"name": "umls", "use_umls": True, "schema_guided": UMLS_SCHEMA_GUIDED, "enabled": run_umls_stage},
]

rows = []
artifact_rows = []

for stage in stages:
    if not stage["enabled"]:
        continue
    os.environ["UMLS_ENABLED"] = "true" if stage["use_umls"] else "false"
    stage_root = output_root / stage["name"]
    artifact_dir = stage_root / "artifacts"
    artifact_dir.mkdir(parents=True, exist_ok=True)

    for generation_model in generation_model_sweep:
        os.environ["INDEX_LLM_MODEL"] = generation_model
        os.environ["VLLM_MODEL"] = generation_model

        for embedding_model in embedding_model_sweep:
            os.environ["INDEX_EMBEDDING_MODEL"] = embedding_model
            os.environ["VLLM_EMBEDDING_MODEL"] = embedding_model

            slug = combo_slug(generation_model, embedding_model)
            index_dir = stage_root / "indexes" / slug
            html_path = artifact_dir / f"{slug}.html"
            jpeg_path = artifact_dir / f"{slug}.jpg"
            results_path = artifact_dir / f"{slug}_results.csv"

            started = time.time()
            indexed_graph = await asyncio.to_thread(
                ensure_index,
                input_dir=mimic_notes_dir,
                output_dir=index_dir,
                use_umls=stage["use_umls"],
                schema_guided=stage["schema_guided"],
            )
            build_seconds = time.time() - started

            graph_html = save_clinical_entity_graph(index_dir, html_path, source="auto")
            graph_jpeg = save_clinical_entity_graph_jpeg(index_dir, jpeg_path, source="auto", title=f"{stage['name']}: {slug}")
            combo_rows = []
            for item in questions:
                prompt = item["question"]
                if item.get("options"):
                    prompt = f"{prompt}\n\n" + format_options(item["options"])
                response, context = await query_index_context(index=indexed_graph, query=prompt)
                predicted = extract_answer(response, item["options"]) if item.get("options") else None
                combo_rows.append({
                    "stage": stage["name"],
                    "id": item.get("id"),
                    "question": item["question"],
                    "gold_answer": item.get("answer"),
                    "predicted": predicted,
                    "model": generation_model,
                    "embedding_model": embedding_model,
                    "combo_slug": slug,
                    "response": response[:1000],
                    "context_preview": context[:1000] if context else None,
                    "html_plot": graph_html.as_posix(),
                    "jpeg_plot": graph_jpeg.as_posix(),
                })

            combo_results = pd.DataFrame(combo_rows)
            combo_results.to_csv(results_path, index=False)
            rows.extend(combo_rows)
            artifact_rows.append({
                "stage": stage["name"],
                "combo_slug": slug,
                "model": generation_model,
                "embedding_model": embedding_model,
                "index_dir": index_dir.as_posix(),
                "html_plot": graph_html.as_posix(),
                "jpeg_plot": graph_jpeg.as_posix(),
                "results_csv": results_path.as_posix(),
                "question_count": len(combo_results),
                "build_seconds": round(build_seconds, 2),
                "umls_enabled": stage["use_umls"],
                "schema_guided": stage["schema_guided"],
            })
            print(f"{stage['name']} / {slug}: built in {build_seconds:.1f}s")

            # The index owns the offline generation and embedding engines. Release it before
            # loading the next combination so a 20 GB GPU does not retain previous models.
            del indexed_graph
            gc.collect()
            try:
                import torch
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            except ImportError:
                pass

results = pd.DataFrame(rows)
artifacts = pd.DataFrame(artifact_rows)
artifacts_path = output_root / "artifact_manifest.csv"
results_path = output_root / "evaluation_results.csv"
artifacts.to_csv(artifacts_path, index=False)
results.to_csv(results_path, index=False)
artifacts


## Inline Results

The cells below show the saved artifacts directly in the notebook: manifest tables, static graph previews, optional interactive graph frames, and comparison charts.


In [ ]:
display_columns = [
    "stage",
    "combo_slug",
    "model",
    "embedding_model",
    "build_seconds",
    "question_count",
    "umls_enabled",
    "schema_guided",
    "html_plot",
    "jpeg_plot",
]
artifact_view = artifacts[[column for column in display_columns if column in artifacts.columns]] if not artifacts.empty else artifacts
artifact_view


In [ ]:
max_graph_previews = NOTEBOOK_GRAPH_PREVIEW_LIMIT
if artifacts.empty:
    print("No graph artifacts to display yet. Run the pipeline cell first.")
else:
    for row in artifacts.head(max_graph_previews).itertuples(index=False):
        title = f"{row.stage} / {row.combo_slug}"
        print(title)
        jpeg_path = Path(row.jpeg_plot)
        if jpeg_path.exists():
            display(Image(filename=str(jpeg_path), width=900))
        else:
            print(f"Missing JPEG preview: {jpeg_path}")


In [ ]:
interactive_index = NOTEBOOK_INTERACTIVE_GRAPH_INDEX
if artifacts.empty:
    print("No interactive graph artifact to display yet.")
else:
    selected = artifacts.iloc[min(interactive_index, len(artifacts) - 1)]
    html_path = Path(selected["html_plot"])
    print(f"Interactive graph: {selected['stage']} / {selected['combo_slug']}")
    if html_path.exists():
        display(IFrame(src=html_path.as_posix(), width="100%", height=720))
    else:
        print(f"Missing HTML graph: {html_path}")


In [ ]:
if artifacts.empty:
    print("No artifact metrics to chart yet.")
else:
    build_chart = artifacts.copy()
    build_chart["label"] = build_chart["stage"] + " | " + build_chart["combo_slug"]
    ax = build_chart.sort_values("build_seconds").plot.barh(
        x="label",
        y="build_seconds",
        figsize=(12, max(4, 0.45 * len(build_chart))),
        legend=False,
        title="Index Build Time by Stage and Model",
    )
    ax.set_xlabel("seconds")
    ax.set_ylabel("")
    display(ax.figure)

if results.empty or "predicted" not in results.columns:
    print("No evaluation predictions to chart yet.")
else:
    scored = results.dropna(subset=["predicted", "gold_answer"]).copy()
    if scored.empty:
        print("Evaluation rows exist, but no scored predictions were available.")
    else:
        scored["correct"] = scored["predicted"].astype(str).str.strip().str.lower() == scored["gold_answer"].astype(str).str.strip().str.lower()
        accuracy = scored.groupby(["stage", "combo_slug"], as_index=False)["correct"].mean()
        accuracy["accuracy"] = accuracy["correct"] * 100
        accuracy["label"] = accuracy["stage"] + " | " + accuracy["combo_slug"]
        ax = accuracy.sort_values("accuracy").plot.barh(
            x="label",
            y="accuracy",
            figsize=(12, max(4, 0.45 * len(accuracy))),
            legend=False,
            title="Evaluation Accuracy by Stage and Model",
        )
        ax.set_xlabel("accuracy (%)")
        ax.set_ylabel("")
        ax.set_xlim(0, 100)
        display(ax.figure)
        accuracy[["stage", "combo_slug", "accuracy"]]


In [ ]:
results
